# Should I Trade Today?

**Run at 3:00–3:15 PM IST. Two cells. Done.**

| Step | What it checks | Time needed |
|------|---------------|-------------|
| Cell 1 | Set your capital | 5 seconds |
| Cell 2 | Fetches S&P 500 + NIFTY scores → prints TRADE or SKIP | ~30 seconds |

**Rule:** If S&P 500 closed positive last night → BUY the list at 3:20 PM. Sell next morning 9:25 AM.  
If S&P 500 closed negative → do nothing today.

---
Strategy win rate: **76%** over 7 years (2019–2026). Trades ~120 days/year.

In [ ]:
CAPITAL = 200_000   # Rs — change this to your actual capital today

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, yfinance as yf
from datetime import date

TOP_N    = 10
LOOKBACK = 20
TODAY    = pd.Timestamp(date.today())

NIFTY50_YF = [
    'ADANIENT.NS','ADANIPORTS.NS','APOLLOHOSP.NS','ASIANPAINT.NS','AXISBANK.NS',
    'BAJAJ-AUTO.NS','BAJFINANCE.NS','BAJAJFINSV.NS','BEL.NS','BHARTIARTL.NS',
    'CIPLA.NS','COALINDIA.NS','DRREDDY.NS','EICHERMOT.NS','ETERNAL.NS',
    'GRASIM.NS','HCLTECH.NS','HDFCBANK.NS','HDFCLIFE.NS','HINDALCO.NS',
    'HINDUNILVR.NS','ICICIBANK.NS','INDIGO.NS','INFY.NS','ITC.NS',
    'JIOFIN.NS','JSWSTEEL.NS','KOTAKBANK.NS','LT.NS','M%26M.NS',
    'MARUTI.NS','MAXHEALTH.NS','NESTLEIND.NS','NTPC.NS','ONGC.NS',
    'POWERGRID.NS','RELIANCE.NS','SBILIFE.NS','SHRIRAMFIN.NS','SBIN.NS',
    'SUNPHARMA.NS','TCS.NS','TATACONSUM.NS','TMPV.NS','TATASTEEL.NS',
    'TECHM.NS','TITAN.NS','TRENT.NS','ULTRACEMCO.NS','WIPRO.NS',
]

# ── 1. S&P 500 filter ────────────────────────────────────────────────────────
sp = yf.download('^GSPC', period='5d', progress=False)['Close'].squeeze().dropna()
sp = sp[sp.index < TODAY]   # only sessions that closed before today

if len(sp) < 2:
    print('ERROR: Could not fetch S&P 500 data.')
else:
    sp_ret  = float(sp.iloc[-1] / sp.iloc[-2] - 1)
    sp_date = sp.index[-1].date()
    sp_pass = sp_ret >= 0

    # ── 2. NIFTY 50 scores ───────────────────────────────────────────────────
    raw   = yf.download(NIFTY50_YF, period='60d', progress=False, auto_adjust=True)

    def clean(df):
        df = df.copy()
        df.columns = [c.replace('.NS','').replace('%26','&') for c in df.columns]
        return df

    close = clean(raw['Close'])
    open_ = clean(raw['Open'])

    # Drop stocks with too much missing data — clean first, then filter
    valid = close.notna().mean() > 0.7
    close = close.loc[:, valid]
    open_ = open_[close.columns]

    scores     = (open_ / close.shift(1) - 1).rolling(LOOKBACK, min_periods=LOOKBACK).mean()
    last_scores = scores.dropna(how='all').iloc[-1].dropna()
    top        = last_scores.nlargest(TOP_N)
    prev_close = close.iloc[-1]
    alloc      = CAPITAL / TOP_N

    # ── 3. Print decision ────────────────────────────────────────────────────
    print(f'Today: {TODAY.date()}')
    print(f'S&P 500 on {sp_date}: {float(sp.iloc[-1]):,.2f}  ({sp_ret:+.2%})')
    print()

    if sp_pass:
        print('  *** TRADE TODAY ***')
        print('  Buy at 3:20 PM  |  Sell tomorrow at 9:25 AM')
    else:
        print('  *** SKIP TODAY — S&P 500 was negative ***')
        print('  Do not buy. Come back tomorrow.')

    print()
    print(f'  {"#":<4} {"Stock":<13} {"20d Score":>10}  {"Prev Close":>11}  {"Qty to buy":>11}')
    print(f'  {"-" * 55}')
    for i, (sym, score) in enumerate(top.items(), 1):
        price = float(prev_close.get(sym, float('nan')))
        qty   = int(alloc / price) if price > 0 else 0
        print(f'  {i:<4} {sym:<13} {score:>+10.4%}  {price:>11,.2f}  {qty:>11}')
    print(f'  {"-" * 55}')
    print(f'  Capital: Rs {CAPITAL:,.0f}   Per stock: Rs {alloc:,.0f}')
    if not sp_pass:
        print()
        print('  (List above is for reference only — do NOT trade today)')